In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *
from test_utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = 'AdultCA'

In [5]:
from folktables import ACSDataSource, ACSIncome

# Initialize the data source for California in 2024
data_source = ACSDataSource(survey_year='2023', horizon='1-Year', survey='person')

# Download and extract the data for California
ca_data = data_source.get_data(states=['CA'], download=True)

# Define the ACSIncome task
features, label, group = ACSIncome.df_to_numpy(ca_data)
feature_names = ACSIncome.features

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

dataset = pd.DataFrame(features, columns=feature_names)
TARGET_COLUMN = 'target'
dataset['target'] = label
dataset['target'] = LabelEncoder().fit_transform(dataset['target'])
target = dataset['target']
datasetX = dataset.drop('target', axis=1)

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.columns.to_list()
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])

    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])

    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances

Accuracy:  0.8063754427390791


* df.iloc[0]: retrieves based on an index iterator from start to bottom.
* df.loc[0]: retrieves based on the index the df has.

In [7]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [ ]:
feat_unique = {}
for col in datasetX.columns:
    feat_unique[col] = len(datasetX[col].unique())
feat_unique
top_3_difficult_to_change_cols = sorted(feat_unique.items(), key=lambda x: x[1])
print(f"Top 3 difficult to change columns: {top_3_difficult_to_change_cols[:3]}")
features_to_vary = [col for col in datasetX.columns if col not in top_3_difficult_to_change_cols]
features_to_vary

Top 3 difficult to change columns: [('SEX', 2), ('MAR', 5), ('COW', 8)]


['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'WKHP', 'SEX', 'RAC1P']

In [ ]:
import warnings
import time
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

import dice_ml
d = dice_ml.Data(dataframe=dataset, continuous_features=numerical, outcome_name=TARGET_COLUMN)
backend = 'sklearn'
m = dice_ml.Model(model=model, backend=backend)

dice_explainers_with_constraints = []
for i in range(4):
    exp_genetic = dice_ml.Dice(d, m, method='genetic')
    dice_exp_genetic = exp_genetic.generate_counterfactuals(
        instances_to_explain, total_CFs=1, desired_class="opposite",
        features_to_vary=features_to_vary)
    dice_explainers_with_constraints.append(dice_exp_genetic)
import os
import pickle
results_dir = f'{UGCE_dir}/results/dice_objects/{datasetName}'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(dice_explainers_with_constraints, open(f'{results_dir}/dice_exp_genetic.pkl', 'wb'))

In [ ]:
import os
import pickle
results_dir = f'{UGCE_dir}/results/dice_objects/adultCA_with_weights'
os.makedirs(results_dir, exist_ok=True)
# Save the object
pickle.dump(dice_exp_genetic, open(f'{results_dir}/dice_exp_genetic.pkl', 'wb'))
pickle.dump(exp_genetic, open(f'{results_dir}/exp_genetic.pkl', 'wb'))

In [ ]:
# Load the object
import pickle
results_dir = f'{UGCE_dir}/results/dice_objects/adultCA_with_weights'
dice_exp_genetic = pickle.load(open(f'{results_dir}/dice_exp_genetic.pkl', 'rb'))
exp_genetic = pickle.load(open(f'{results_dir}/exp_genetic.pkl', 'rb'))
dice_explainers_with_constraints.append(dice_exp_genetic)

In [12]:
from test_utils import *
aggregate_results_DICE_baseline(iea,instances_to_explain, [dice_exp_genetic], TARGET_COLUMN)

Full Time: mean = 22.2984, std = 0.0000
Generations: mean = 6.0398, std = 0.0000
Coverage: mean = 1.1186, std = 0.0000
Proximity Loss: mean = 0.0691, std = 0.0000
Sparsity: mean = 0.0384, std = 0.0000


# UGCE

## From Scratch

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

constraints = {
    "SEX": '-',
    'MAR': '-',
    'COW': '-',
}

results_baseline_explainer = []
for i in range(5):
    strategy = "fix_population_update_fitness"
    results_baseline = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=False, constraints=constraints,
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=1,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=20, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints={}, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric='weighted_l1',
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_baseline_explainer.append(results_baseline)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/baseline'
os.makedirs(results_dir, exist_ok=True)
strategy = "fix_population_update_fitness"
pickle.dump(results_baseline_explainer, open(f'{results_dir}/results_baseline{strategy}.pkl', 'wb'))

100%|██████████| 20203/20203 [55:25<00:00,  6.08it/s] 


In [9]:
## load the results
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/baseline'
strategy = "fix_population_update_fitness"
results_baseline_explainer = pickle.load(open(f'{results_dir}/results_baseline{strategy}.pkl', 'rb'))

In [15]:
from test_utils import aggregate_results_baseline
aggregate_results_baseline(iea, results_baseline_explainer)

Full Time: mean = 58.3545, std = 2.1228
Generations: mean = 6.0309, std = 0.0013
Coverage: mean = 99.9901, std = 0.0000
Proximity Loss: mean = 0.0518, std = 0.0003
Sparsity: mean = 0.0532, std = 0.0001


## Dynamic

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    "SEX": 'i',
    'MAR': 'i',
    'COW': 'i',
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer = []
for i in range(5):
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=1,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental, open(f'{results_dir}/results_incremental{strategy}.pkl', 'wb'))

100%|██████████| 20203/20203 [1:13:37<00:00,  4.57it/s]

Empty intermediate counter: 0


In [9]:
# load the results
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
strategy = "fix_population_update_fitness"
results_incremental_explainer = pickle.load(open(f'{results_dir}/results_incremental{strategy}.pkl', 'rb'))

In [ ]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer, verbose=True)

Time taken for generating counterfactuals using UGCE dynamic:  20.597366642951965  minutes
Time taken for generating counterfactuals using UGCE dynamic:  20.829141982396443  minutes
Time taken for generating counterfactuals using UGCE dynamic:  22.297064904371897  minutes
Time taken for generating counterfactuals using UGCE dynamic:  22.964952099323273  minutes
Time taken for generating counterfactuals using UGCE dynamic:  21.39011058410009  minutes
Full Time: mean = 21.62, std = 0.89
Generations: mean = 6.02, std = 0.00
Coverage: mean = 57.35, std = 0.30
Proximity Loss: mean = 0.07, std = 0.00
Sparsity: mean = 0.06, std = 0.00
Intermediate Best Distances: mean = 0.14, std = 0.00


(21.61572724262873,
 6.017623273061275,
 57.35313041326405,
 0.06662213527241465,
 0.05838723723724524,
 0.13624443683950763)

# Assess the statistical Importance of correcting the population rather than starting with a random population

## Fixed population

In [12]:
total_time_dynamic_arr_for_ttest=[]
total_generations_arr_for_ttest=[]
total_cfes_found_arr_for_ttest=[]
total_proximity_loss_arr_for_ttest=[]
total_sparsity_arr_for_ttest=[]
total_best_intermediate_best_dist_arr_for_ttest=[]

full_times = []
coverages = []
distances = []
l1s = []
proximities = []
sparsities = []
generation_counts = []
intermediate_best_distances = []

for explainer in results_incremental_explainer:
    forttest, foravgs = stats_incremental(iea, explainer, return_matrices=True)
    time_dynamic_arr, generations_arr, cfes_found_arr, proximity_loss_arr, sparsity_arr, best_intermediate_best_dist_arr = forttest
    time_dynamic, avg_generations, avg_cfes_found, avg_l2, avg_l1, avg_proximity_loss, avg_sparsity, avg_best_intermediate_best_dist = foravgs
    
    total_time_dynamic_arr_for_ttest.extend(time_dynamic_arr)
    total_generations_arr_for_ttest.extend(generations_arr)
    total_cfes_found_arr_for_ttest.extend(cfes_found_arr)
    total_proximity_loss_arr_for_ttest.extend(proximity_loss_arr)
    total_sparsity_arr_for_ttest.extend(sparsity_arr)
    total_best_intermediate_best_dist_arr_for_ttest.extend(best_intermediate_best_dist_arr)

    full_times.append(time_dynamic)
    coverages.append(avg_cfes_found)
    distances.append(avg_l2)
    l1s.append(avg_l1)
    proximities.append(avg_proximity_loss)
    sparsities.append(avg_sparsity)
    generation_counts.append(avg_generations)
    intermediate_best_distances.append(avg_best_intermediate_best_dist)

def print_metric_stats(name, values):
        print(f"{name}: mean = {np.mean(values):.4f}, std = {np.std(values):.4f}")

## Random population

In [7]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    "SEX": 'i',
    'MAR': 'i',
    'COW': 'i',
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_random = []
for i in range(5):
    import time
    strategy = "new_random_population"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="new_random_population", population_size_dynamic=0, cfes_requested=1,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_random.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_random, open(f'{results_dir}/results_incremental{strategy}.pkl', 'wb'))

100%|██████████| 20203/20203 [1:18:43<00:00,  4.28it/s]


Empty intermediate counter: 0


100%|██████████| 20203/20203 [1:20:31<00:00,  4.18it/s]


Empty intermediate counter: 0


100%|██████████| 20203/20203 [1:20:25<00:00,  4.19it/s]


Empty intermediate counter: 0


100%|██████████| 20203/20203 [1:20:43<00:00,  4.17it/s]


Empty intermediate counter: 0


100%|██████████| 20203/20203 [1:20:28<00:00,  4.18it/s]


Empty intermediate counter: 0


In [ ]:
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
strategy = "new_random_population"
results_incremental_explainer_random = pickle.load(open(f'{results_dir}/results_incremental{strategy}.pkl', 'rb'))

In [10]:
total_time_dynamic_arr_for_ttest_random=[]
total_generations_arr_for_ttest_random=[]
total_cfes_found_arr_for_ttest_random=[]
total_proximity_loss_arr_for_ttest_random=[]
total_sparsity_arr_for_ttest_random=[]
total_best_intermediate_best_dist_arr_for_ttest_random=[]

full_times_random = []
coverages_random = []
distances_random = []
l1s_random = []
proximities_random = []
sparsities_random = []
generation_counts_random = []
intermediate_best_distances_random = []

for explainer in results_incremental_explainer_random:
    forttest, foravgs = stats_incremental(iea, explainer, return_matrices=True)
    time_dynamic_arr, generations_arr, cfes_found_arr, proximity_loss_arr, sparsity_arr, best_intermediate_best_dist_arr = forttest
    time_dynamic, avg_generations, avg_cfes_found, avg_l2, avg_l1, avg_proximity_loss, avg_sparsity, avg_best_intermediate_best_dist = foravgs
    
    total_time_dynamic_arr_for_ttest_random.extend(time_dynamic_arr)
    total_generations_arr_for_ttest_random.extend(generations_arr)
    total_cfes_found_arr_for_ttest_random.extend(cfes_found_arr)
    total_proximity_loss_arr_for_ttest_random.extend(proximity_loss_arr)
    total_sparsity_arr_for_ttest_random.extend(sparsity_arr)
    total_best_intermediate_best_dist_arr_for_ttest_random.extend(best_intermediate_best_dist_arr)

    full_times_random.append(time_dynamic)
    coverages_random.append(avg_cfes_found)
    distances_random.append(avg_l2)
    l1s_random.append(avg_l1)
    proximities_random.append(avg_proximity_loss)
    sparsities_random.append(avg_sparsity)
    generation_counts_random.append(avg_generations)
    intermediate_best_distances_random.append(avg_best_intermediate_best_dist)

def print_metric_stats(name, values):
        print(f"{name}: mean = {np.mean(values):.4f}, std = {np.std(values):.4f}")

In [ ]:
from scipy.stats import ttest_ind
import numpy as np

from scipy.stats import ttest_ind
import numpy as np

def format_stat(value):
    """Format a statistic to 2 decimal places"""
    return f"{value:.2f}"

def format_pvalue(pvalue):
    """Format p-value with scientific notation"""
    if pvalue < 1e-100:
        return "0.0\\times 10^{0}"
    
    sci_notation = f"{pvalue:.1e}".split('e')
    base = float(sci_notation[0])
    exponent = int(sci_notation[1])
    
    return f"{base:.1f}\\times 10^{{{exponent}}}"


# Perform t-tests 
ttest_times = ttest_ind(total_time_dynamic_arr_for_ttest, total_time_dynamic_arr_for_ttest_random)
ttest_generations = ttest_ind(total_generations_arr_for_ttest, total_generations_arr_for_ttest_random)
ttest_cfes_found = ttest_ind(total_cfes_found_arr_for_ttest, total_cfes_found_arr_for_ttest_random)
ttest_distances = ttest_ind(total_proximity_loss_arr_for_ttest, total_proximity_loss_arr_for_ttest_random)
ttest_sparsity = ttest_ind(total_sparsity_arr_for_ttest, total_sparsity_arr_for_ttest_random)
test_distances_intermediate = ttest_ind(total_best_intermediate_best_dist_arr_for_ttest, total_best_intermediate_best_dist_arr_for_ttest_random)

warm_means = [
    np.mean(full_times),
    np.mean(generation_counts), 
    np.mean(coverages),
    np.mean(proximities),
    np.mean(sparsities),
    # np.mean(intermediate_best_distances)
]

random_means = [
    np.mean(full_times_random),
    np.mean(generation_counts_random),
    np.mean(coverages_random),
    np.mean(proximities_random),
    np.mean(sparsities_random),
    # np.mean(intermediate_best_distances_random),
]

pvalues = [
    ttest_times.pvalue,
    ttest_generations.pvalue, 
    ttest_cfes_found.pvalue,
    ttest_distances.pvalue,
    ttest_sparsity.pvalue,
    # test_distances_intermediate.pvalue
]

rows = [
    f"\\shortstack{{\\texttt{{Compas}}}} & {' & '.join(f'${format_pvalue(p)}$' for p in pvalues)} \\\\",
    f"Warm & {' & '.join(f'${format_stat(m)}$' for m in warm_means)} \\\\",
    f"Random & {' & '.join(f'${format_stat(m)}$' for m in random_means)} \\\\"
]

print("\n".join(rows))

\shortstack{\texttt{Compas}} & $0.0\times 10^{0}$ & $1.2\times 10^{-75}$ & $5.2\times 10^{-1}$ & $0.0\times 10^{0}$ & $0.0\times 10^{0}$ \\
Warm & $21.62$ & $6.02$ & $57.35$ & $0.07$ & $0.06$ \\
Random & $29.08$ & $6.10$ & $57.48$ & $0.08$ & $0.05$ \\
